## 02 — Preprocessing

1. Load and concatenate all 4 seasons of outfield and GK parquets, add season column
2. Fix team name inconsistencies via TEAM_NAME_FIX dictionary
3. Add match context features: home_away, opponent, team_goals_scored, opp_goals_scored, result
4. Audit null rates and drop columns above 50% null threshold with manual exceptions
5. Drop redundant columns (xg_and_xa, expected_goals_non_penalty, duel_won, interceptions for GK)
6. Replace 0.01 placeholder values with 0
7. Drop rows with null rating_title
8. Rename dirty column matchstats.headers.tackles → tackles
9. Fill remaining nulls with 0
10. Filter out players with fewer than 30 minutes played
11. Save outfield_clean.parquet and gk_clean.parquet

In [168]:
import pandas as pd
from pathlib import Path

outfield_dfs = []
gk_dfs = []

for season_dir in sorted(Path('data/47').iterdir()):
    out_path = season_dir / 'output' / 'outfield_players.parquet'
    gk_path  = season_dir / 'output' / 'goalkeepers.parquet'

    if out_path.exists():
        df = pd.read_parquet(out_path)
        df['season'] = season_dir.name
        outfield_dfs.append(df)
        print(f'Outfield {season_dir.name}: {df.shape[0]} rows x {df.shape[1]} cols')

    if gk_path.exists():
        df = pd.read_parquet(gk_path)
        df['season'] = season_dir.name
        gk_dfs.append(df)
        print(f'GK      {season_dir.name}: {df.shape[0]} rows x {df.shape[1]} cols')

    print()

outfield = pd.concat(outfield_dfs, ignore_index=True)
gk       = pd.concat(gk_dfs,       ignore_index=True)

print(f'Total outfield : {outfield.shape[0]:,} rows x {outfield.shape[1]} cols')
print(f'Total GK       : {gk.shape[0]:,} rows x {gk.shape[1]} cols')

Outfield 2021_2022: 7812 rows x 68 cols
GK      2021_2022: 760 rows x 53 cols

Outfield 2022_2023: 7954 rows x 68 cols
GK      2022_2023: 760 rows x 53 cols

Outfield 2023_2024: 7733 rows x 68 cols
GK      2023_2024: 762 rows x 54 cols

Outfield 2024_2025: 7598 rows x 68 cols
GK      2024_2025: 760 rows x 53 cols

Total outfield : 31,097 rows x 68 cols
Total GK       : 3,042 rows x 54 cols


In [169]:
if 'goals' in gk.columns:
    gk_goals = gk[['match_id', 'team_name', 'goals']]
else:
    gk_goals = gk[['match_id', 'team_name']].assign(goals=0)
    print('GK goals column not found — defaulting to 0 for all GKs')

all_players = pd.concat([
    outfield[['match_id', 'team_name', 'goals']],
    gk_goals
], ignore_index=True)

print('=== Matches per season in outfield data ===')
print(outfield.groupby('season')['match_id'].nunique())

print('\n=== Matches per season in team_goals ===')
# Add season to team_goals for checking
team_goals_check = (
    all_players
    .merge(outfield[['match_id', 'season']].drop_duplicates(), on='match_id', how='left')
    .groupby('season')['match_id']
    .nunique()
)
print(team_goals_check)

# Check if any match has only 1 team recorded instead of 2
match_team_counts = all_players.groupby('match_id')['team_name'].nunique()
single_team_matches = match_team_counts[match_team_counts < 2]
print(f'\nMatches with only 1 team recorded : {len(single_team_matches)}')
print(f'Matches with 2 teams recorded     : {(match_team_counts == 2).sum()}')

# Sample some single team matches
if len(single_team_matches) > 0:
    sample_ids = single_team_matches.index[:5]
    print(f'\nSample single-team match IDs: {list(sample_ids)}')
    print(outfield[outfield['match_id'].isin(sample_ids)][['match_id', 'home_team', 'away_team', 'team_name']].drop_duplicates().to_string(index=False))

GK goals column not found — defaulting to 0 for all GKs
=== Matches per season in outfield data ===
season
2021_2022    380
2022_2023    380
2023_2024    381
2024_2025    380
Name: match_id, dtype: int64

=== Matches per season in team_goals ===
season
2021_2022    380
2022_2023    380
2023_2024    381
2024_2025    380
Name: match_id, dtype: int64

Matches with only 1 team recorded : 0
Matches with 2 teams recorded     : 1521


In [170]:
# Get all unique team names from every possible column
team_names_from_team_name  = set(outfield['team_name'].unique()) | set(gk['team_name'].unique())
team_names_from_home_away  = set(outfield['home_team'].unique()) | set(outfield['away_team'].unique()) | \
                             set(gk['home_team'].unique())       | set(gk['away_team'].unique())

print('=== Teams in home_team/away_team but NOT in team_name ===')
mismatch_1 = team_names_from_home_away - team_names_from_team_name
for t in sorted(mismatch_1):
    print(f'  "{t}"')

print(f'\n=== Teams in team_name but NOT in home_team/away_team ===')
mismatch_2 = team_names_from_team_name - team_names_from_home_away
for t in sorted(mismatch_2):
    print(f'  "{t}"')

print(f'\nTotal mismatches : {len(mismatch_1) + len(mismatch_2)}')
print(f'\nAll unique team names in team_name column ({len(team_names_from_team_name)}):')
for t in sorted(team_names_from_team_name):
    print(f'  "{t}"')

=== Teams in home_team/away_team but NOT in team_name ===
  "AFC Bournemouth"
  "Brighton & Hove Albion"

=== Teams in team_name but NOT in home_team/away_team ===
  "Bournemouth"
  "Brighton and Hove Albion"

Total mismatches : 4

All unique team names in team_name column (26):
  "Arsenal"
  "Aston Villa"
  "Bournemouth"
  "Brentford"
  "Brighton and Hove Albion"
  "Burnley"
  "Chelsea"
  "Crystal Palace"
  "Everton"
  "Fulham"
  "Ipswich Town"
  "Leeds United"
  "Leicester City"
  "Liverpool"
  "Luton Town"
  "Manchester City"
  "Manchester United"
  "Newcastle United"
  "Norwich City"
  "Nottingham Forest"
  "Sheffield United"
  "Southampton"
  "Tottenham Hotspur"
  "Watford"
  "West Ham United"
  "Wolverhampton Wanderers"


In [171]:
# Normalize team names to fix & vs and inconsistency
TEAM_NAME_FIX = {
    'Brighton & Hove Albion' : 'Brighton and Hove Albion',
    'AFC Bournemouth'        : 'Bournemouth',
}

# Apply to home_team, away_team and opponent columns on both dataframes
for df in [outfield, gk]:
    df['home_team'] = df['home_team'].replace(TEAM_NAME_FIX)
    df['away_team'] = df['away_team'].replace(TEAM_NAME_FIX)

In [172]:
# home_away binary feature: 1 if home, 0 if away
outfield['home_away'] = (outfield['team_name'] == outfield['home_team']).astype(int)
gk['home_away']       = (gk['team_name'] == gk['home_team']).astype(int)

# opponent column: the other team in the match
outfield['opponent'] = outfield.apply(
    lambda r: r['away_team'] if r['team_name'] == r['home_team'] else r['home_team'], axis=1
)
gk['opponent'] = gk.apply(
    lambda r: r['away_team'] if r['team_name'] == r['home_team'] else r['home_team'], axis=1
)

print('\n=== home_away distribution ===')
print(f'Outfield — home: {(outfield["home_away"]==1).sum():,} | away: {(outfield["home_away"]==0).sum():,}')
print(f'GK       — home: {(gk["home_away"]==1).sum():,} | away: {(gk["home_away"]==0).sum():,}')


=== home_away distribution ===
Outfield — home: 15,544 | away: 15,553
GK       — home: 1,521 | away: 1,521


In [173]:
# Team goals per match — combine outfield + GK to get full tally
# Use raw data so no goals are missed from filtered players
if 'goals' in gk.columns:
    gk_goals = gk[['match_id', 'team_name', 'goals']]
else:
    gk_goals = gk[['match_id', 'team_name']].assign(goals=0)
    print('GK goals column not found — defaulting to 0 for all GKs')

all_players = pd.concat([
    outfield[['match_id', 'team_name', 'goals']],
    gk_goals
], ignore_index=True)

# Fill null goals with 0 just for this aggregation
all_players['goals'] = all_players['goals'].fillna(0)

team_goals = (
    all_players
    .groupby(['match_id', 'team_name'])['goals']
    .sum()
    .reset_index()
    .rename(columns={'goals': 'team_goals_scored'})
)

print(f'Team goals computed for {team_goals["match_id"].nunique():,} matches')

# Merge team goals back
outfield = outfield.merge(team_goals, on=['match_id', 'team_name'], how='left')
gk       = gk.merge(team_goals, on=['match_id', 'team_name'], how='left')

# Opponent goals
outfield = outfield.merge(
    team_goals.rename(columns={'team_name': 'opponent', 'team_goals_scored': 'opp_goals_scored'}),
    on=['match_id', 'opponent'], how='left'
)
gk = gk.merge(
    team_goals.rename(columns={'team_name': 'opponent', 'team_goals_scored': 'opp_goals_scored'}),
    on=['match_id', 'opponent'], how='left'
)

def get_result(row):
    if row['team_goals_scored'] > row['opp_goals_scored']:
        return 'win'
    elif row['team_goals_scored'] < row['opp_goals_scored']:
        return 'loss'
    else:
        return 'draw'

outfield['result'] = outfield.apply(get_result, axis=1)
gk['result']       = gk.apply(get_result, axis=1)

print('\n=== result distribution ===')
print(f'Outfield:\n{outfield["result"].value_counts()}')
print(f'\nGK:\n{gk["result"].value_counts()}')

print('\n=== score check — sample matches ===')
sample = outfield[['match_id', 'home_team', 'away_team', 'team_name',
                   'team_goals_scored', 'opp_goals_scored', 'result']].drop_duplicates('match_id').head(10)
print(sample.to_string(index=False))

print(f'\nAny nulls in new cols (outfield): {outfield[["home_away","opponent","result","team_goals_scored","opp_goals_scored"]].isnull().sum().sum()}')
print(f'Any nulls in new cols (GK)      : {gk[["home_away","opponent","result","team_goals_scored","opp_goals_scored"]].isnull().sum().sum()}')

GK goals column not found — defaulting to 0 for all GKs
Team goals computed for 1,521 matches

=== result distribution ===
Outfield:
result
loss    10933
win     10928
draw     9236
Name: count, dtype: int64

GK:
result
loss    1069
win     1069
draw     904
Name: count, dtype: int64

=== score check — sample matches ===
 match_id         home_team                away_team                team_name  team_goals_scored  opp_goals_scored result
  3609929         Brentford                  Arsenal                  Arsenal                1.0               1.0   draw
  3609930           Burnley Brighton and Hove Albion Brighton and Hove Albion                2.0               1.0    win
  3609931           Chelsea           Crystal Palace           Crystal Palace                1.0               3.0   loss
  3609932           Everton              Southampton                  Everton                2.0               0.0    win
  3609933    Leicester City  Wolverhampton Wanderers  Wolverhampton

In [174]:
# Separate the ID columns from the stat columns
ID_COLS = [
    'match_id', 'round', 'match_date', 'home_team', 'away_team',
    'player_id', 'player_name', 'team_id', 'team_name',  'minutes_played',
    'shirt_number', 'position_id', 'is_goalkeeper', 'season',
    # match context features
    'home_away', 'opponent', 'team_goals_scored', 'opp_goals_scored', 'result'
]

outfield_stat_cols = [c for c in outfield.columns if c not in ID_COLS]
gk_stat_cols       = [c for c in gk.columns if c not in ID_COLS]

print(f'Outfield stat columns ({len(outfield_stat_cols)}):')
for c in outfield_stat_cols:
    print(f'  {c}')

print(f'\nGK stat columns ({len(gk_stat_cols)}):')
for c in gk_stat_cols:
    print(f'  {c}')

Outfield stat columns (54):
  ShotsOffTarget
  ShotsOnTarget
  accurate_passes
  assists
  chances_created
  defensive_actions
  expected_assists
  expected_goals
  expected_goals_on_target_variant
  goals
  rating_title
  shot_accuracy
  xg_and_xa
  blocked_shots
  big_chance_created_team_title
  errors_led_to_goal
  conceded_penalties
  penalties_won
  owngoal
  missed_penalty
  accurate_crosses
  corners
  dispossessed
  dribbles_succeeded
  expected_goals_non_penalty
  long_balls_accurate
  passes_into_final_third
  touches
  touches_opp_box
  Offsides
  big_chance_missed_title
  shots_woodwork
  clearances
  dribbled_past
  headed_clearance
  interceptions
  matchstats.headers.tackles
  recoveries
  shot_blocks
  last_man_tackle
  clearance_off_the_line
  aerials_won
  duel_lost
  duel_won
  fouls
  ground_duels_won
  was_fouled
  accurate_crosses_total
  accurate_passes_total
  aerials_won_total
  dribbles_succeeded_total
  ground_duels_won_total
  long_balls_accurate_total
  sho

In [175]:
print('=== OUTFIELD NULL RATES ===')
null_rates = outfield[outfield_stat_cols].isnull().mean().sort_values(ascending=False)
print(null_rates[null_rates > 0].to_string())

print('\n=== GK NULL RATES ===')
gk_null_rates = gk[gk_stat_cols].isnull().mean().sort_values(ascending=False)
print(gk_null_rates[gk_null_rates > 0].to_string())

print('\n=== OUTFIELD BASIC STATS (non-null counts + mean) ===')
print(outfield[outfield_stat_cols].describe().loc[['count','mean','min','max']].T.to_string())

print('\n=== GK BASIC STATS (non-null counts + mean) ===')
print(gk[gk_stat_cols].describe().loc[['count','mean','min','max']].T.to_string())

=== OUTFIELD NULL RATES ===
missed_penalty                      0.997942
owngoal                             0.995787
penalties_won                       0.990160
conceded_penalties                  0.988520
clearance_off_the_line              0.987684
errors_led_to_goal                  0.987008
last_man_tackle                     0.985208
shots_woodwork                      0.976879
big_chance_missed_title             0.894009
Offsides                            0.887738
big_chance_created_team_title       0.871145
corners                             0.832846
blocked_shots                       0.741165
expected_goals_on_target_variant    0.723864
shot_accuracy_total                 0.525742
shot_accuracy                       0.525742
headed_clearance                    0.490272
accurate_crosses_total              0.480947
accurate_crosses                    0.480947
was_fouled                          0.462167
expected_goals_non_penalty          0.427212
ShotsOffTarget             

In [176]:
# Auto-drop above 50% null
drop_candidates = null_rates[null_rates > 0.50].index.tolist()

# Keep blocked_shots despite >50% null — min value is 1, real event, not redundant
drop_candidates = [c for c in drop_candidates if c != 'blocked_shots']

# Add redundant columns manually
REDUNDANT = ['xg_and_xa', 'expected_goals_non_penalty', 'duel_won']
drop_candidates = list(set(drop_candidates + REDUNDANT))

print(f"Dropping {len(drop_candidates)} columns:")
for c in sorted(drop_candidates):
    print(f"  {c}")

# Create cleaned dataframe
outfield_clean = outfield.drop(columns=drop_candidates)

print(f"\noutfield shape before : {outfield.shape}")
print(f"outfield shape after  : {outfield_clean.shape}")

Dropping 18 columns:
  Offsides
  big_chance_created_team_title
  big_chance_missed_title
  clearance_off_the_line
  conceded_penalties
  corners
  duel_won
  errors_led_to_goal
  expected_goals_non_penalty
  expected_goals_on_target_variant
  last_man_tackle
  missed_penalty
  owngoal
  penalties_won
  shot_accuracy
  shot_accuracy_total
  shots_woodwork
  xg_and_xa

outfield shape before : (31097, 73)
outfield shape after  : (31097, 55)


In [177]:
# Auto-drop above 50% null
gk_drop_candidates = gk_null_rates[gk_null_rates > 0.50].index.tolist()

# Keep aerials_won and aerials_won_total despite >50% null — sweeper keeper behavior
gk_drop_candidates = [c for c in gk_drop_candidates if c not in ['aerials_won', 'aerials_won_total']]

# Keep ground_duels_won and ground_duels_won_total despite >50% null — same reason
gk_drop_candidates = [c for c in gk_drop_candidates if c not in ['ground_duels_won', 'ground_duels_won_total']]

# Add near-zero and redundant columns manually
REDUNDANT_GK = ['interceptions']
gk_drop_candidates = list(set(gk_drop_candidates + REDUNDANT_GK))

print(f"Dropping {len(gk_drop_candidates)} columns:")
for c in sorted(gk_drop_candidates):
    print(f"  {c}")

# Create cleaned dataframe
gk_clean = gk.drop(columns=gk_drop_candidates)

print(f"\ngk shape before : {gk.shape}")
print(f"gk shape after  : {gk_clean.shape}")

Dropping 14 columns:
  assists
  chances_created
  conceded_penalties
  dribbles_succeeded
  dribbles_succeeded_total
  duel_lost
  duel_won
  errors_led_to_goal
  expected_assists
  interceptions
  owngoal
  saved_penalties
  was_fouled
  xg_and_xa

gk shape before : (3042, 59)
gk shape after  : (3042, 45)


In [178]:
# Replace 0.01 placeholders
PLACEHOLDER_COLS_OUTFIELD = ['expected_goals', 'expected_assists']
PLACEHOLDER_COLS_GK       = ['expected_goals_on_target_faced']

for col in PLACEHOLDER_COLS_OUTFIELD:
    if col in outfield_clean.columns:
        mask = outfield_clean[col] == 0.01
        outfield_clean.loc[mask, col] = 0
        print(f'Outfield {col:<35} replaced {mask.sum()} placeholder 0.01s')

for col in PLACEHOLDER_COLS_GK:
    if col in gk_clean.columns:
        mask = gk_clean[col] == 0.01
        gk_clean.loc[mask, col] = 0
        print(f'GK      {col:<35} replaced {mask.sum()} placeholder 0.01s')

Outfield expected_goals                      replaced 431 placeholder 0.01s
Outfield expected_assists                    replaced 6451 placeholder 0.01s
GK      expected_goals_on_target_faced      replaced 3 placeholder 0.01s


In [179]:
# Drop null rating_title BEFORE fillna so we don't fill it with 0
outfield_before = len(outfield_clean)
gk_before       = len(gk_clean)
outfield_clean  = outfield_clean.dropna(subset=['rating_title'])
gk_clean        = gk_clean.dropna(subset=['rating_title'])
print(f'\nDropped {outfield_before - len(outfield_clean)} outfield rows with null rating_title')
print(f'Dropped {gk_before - len(gk_clean)} GK rows with null rating_title')


Dropped 231 outfield rows with null rating_title
Dropped 0 GK rows with null rating_title


In [180]:
# Rename dirty column BEFORE computing stat cols
outfield_clean = outfield_clean.rename(columns={'matchstats.headers.tackles': 'tackles'})
gk_clean       = gk_clean.rename(columns={'matchstats.headers.tackles': 'tackles'})

# Recompute stat cols AFTER rename so the list is accurate
outfield_stat_cols = [c for c in outfield_clean.columns if c not in ID_COLS]
gk_stat_cols       = [c for c in gk_clean.columns if c not in ID_COLS]

In [181]:
outfield_nulls_before = outfield_clean[outfield_stat_cols].isnull().sum()
gk_nulls_before       = gk_clean[gk_stat_cols].isnull().sum()

# Fill remaining nulls with 0
outfield_clean[outfield_stat_cols] = outfield_clean[outfield_stat_cols].fillna(0)
gk_clean[gk_stat_cols]             = gk_clean[gk_stat_cols].fillna(0)

print('\n=== OUTFIELD — nulls filled with 0 ===')
outfield_filled = outfield_nulls_before[outfield_nulls_before > 0].sort_values(ascending=False)
for col, count in outfield_filled.items():
    pct = count / len(outfield_clean) * 100
    print(f'  {col:<40} {count:>5} rows filled ({pct:.1f}%)')
print(f'  Total: {outfield_filled.sum():,} values filled across {len(outfield_filled)} columns')

print('\n=== GK — nulls filled with 0 ===')
gk_filled = gk_nulls_before[gk_nulls_before > 0].sort_values(ascending=False)
for col, count in gk_filled.items():
    pct = count / len(gk_clean) * 100
    print(f'  {col:<40} {count:>5} rows filled ({pct:.1f}%)')
print(f'  Total: {gk_filled.sum():,} values filled across {len(gk_filled)} columns')



=== OUTFIELD — nulls filled with 0 ===
  blocked_shots                            22831 rows filled (74.0%)
  headed_clearance                         15027 rows filled (48.7%)
  accurate_crosses_total                   14756 rows filled (47.8%)
  accurate_crosses                         14756 rows filled (47.8%)
  was_fouled                               14155 rows filled (45.9%)
  ShotsOffTarget                           13086 rows filled (42.4%)
  ShotsOnTarget                            13086 rows filled (42.4%)
  expected_goals                           13031 rows filled (42.2%)
  dribbles_succeeded                       13003 rows filled (42.1%)
  dribbles_succeeded_total                 13003 rows filled (42.1%)
  long_balls_accurate_total                 5910 rows filled (19.1%)
  long_balls_accurate                       5910 rows filled (19.1%)
  expected_assists                          4930 rows filled (16.0%)
  passes_into_final_third                   2973 rows filled (9

In [182]:
# Filter minimum minutes < 30 to remove players with limited playing time that could skew the stats
outfield_before = len(outfield_clean)
gk_before       = len(gk_clean)

outfield_clean = outfield_clean[outfield_clean['minutes_played'] >= 30]
gk_clean       = gk_clean[gk_clean['minutes_played'] >= 30]

print(f'Outfield rows removed (< 30 mins): {outfield_before - len(outfield_clean):>5} ({(outfield_before - len(outfield_clean)) / outfield_before * 100:.1f}%)')
print(f'GK rows removed (< 30 mins)      : {gk_before - len(gk_clean):>5} ({(gk_before - len(gk_clean)) / gk_before * 100:.1f}%)')
print(f'\nOutfield remaining : {len(outfield_clean):,} rows')
print(f'GK remaining       : {len(gk_clean):,} rows')

Outfield rows removed (< 30 mins):   537 (1.7%)
GK rows removed (< 30 mins)      :     0 (0.0%)

Outfield remaining : 30,329 rows
GK remaining       : 3,042 rows


In [183]:
print(f'\nOutfield final shape : {outfield_clean.shape}')
print(f'GK final shape       : {gk_clean.shape}')
print(f'\nAny nulls in outfield stat cols : {outfield_clean[outfield_stat_cols].isnull().any().any()}')
print(f'Any nulls in GK stat cols       : {gk_clean[gk_stat_cols].isnull().any().any()}')
print(f'\nOutfield rating_title zeros : {(outfield_clean["rating_title"] == 0).sum()}')
print(f'GK rating_title zeros       : {(gk_clean["rating_title"] == 0).sum()}')
print(f'\nOutfield rating_title stats:\n{outfield_clean["rating_title"].describe()}')


Outfield final shape : (30329, 55)
GK final shape       : (3042, 45)

Any nulls in outfield stat cols : False
Any nulls in GK stat cols       : False

Outfield rating_title zeros : 0
GK rating_title zeros       : 0

Outfield rating_title stats:
count    30329.000000
mean         7.018574
std          0.804080
min          3.150000
25%          6.470000
50%          7.010000
75%          7.550000
max          9.890000
Name: rating_title, dtype: float64


In [184]:
from pathlib import Path

PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Export cleaned dataframes
outfield_clean.to_parquet(PROCESSED_DIR / 'outfield_clean.parquet', index=False)
gk_clean.to_parquet(PROCESSED_DIR / 'gk_clean.parquet', index=False)

outfield_clean.to_csv(PROCESSED_DIR / 'outfield_clean.csv', index=False)
gk_clean.to_csv(PROCESSED_DIR / 'gk_clean.csv', index=False)

print(f'Saved to {PROCESSED_DIR.resolve()}')
print(f'  outfield_clean.parquet — {len(outfield_clean):,} rows x {outfield_clean.shape[1]} cols')
print(f'  gk_clean.parquet       — {len(gk_clean):,} rows x {gk_clean.shape[1]} cols')

Saved to C:\Users\natmaw\NatMaws Documents\Boston Stuff\EECE 5644 Machine Learning and Pattern Recognition\Final Project\GOALS-Game_Outcome_and_Analytics_Learning_System-\data\processed
  outfield_clean.parquet — 30,329 rows x 55 cols
  gk_clean.parquet       — 3,042 rows x 45 cols
